### Libraries



In [1]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import hf_hub_download, login
import pandas as pd
from sklearn.metrics import classification_report
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.3 MB/s eta 0:00:00


### Login to huggingface

In [2]:
login()

### Testing

In [6]:
# === Use GPU if available ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# === Load tokenizer and model with LoRA ===
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-checkpoints-sentiment"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(base_model, lora_repo_id)
model = model.to(device)
model.eval()

# === Download and load GermEval CSV from Hugging Face ===
csv_path = hf_hub_download(
    repo_id="eduhuemar001/sentiment-GermEval2017",
    filename="germeval2017_cleaned.csv",
    repo_type="dataset"
)

df = pd.read_csv(csv_path)
df = df[["review_text", "sentiment"]]
df = df.dropna(subset=["review_text", "sentiment"])
df["review_text"] = df["review_text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["sentiment"].isin(["positive", "neutral", "negative"])]

class_counts = df["sentiment"].value_counts()
print("Available samples per class:", class_counts.to_dict())

# Only use classes that have at least 1 sample
available_classes = ["positiv", "neutral", "negativ"]
available_classes = [cls for cls in available_classes if class_counts.get(cls, 0) > 0]

# Find the smallest available count among the valid classes
if not available_classes:
    raise ValueError("No valid sentiment classes found in the dataset.")

min_samples = min([class_counts[cls] for cls in available_classes])
print(f"Sampling {min_samples} examples per class:", available_classes)

# Sample equally across all available classes
df_sampled = pd.concat([
    df[df["sentiment"] == cls].sample(min_samples, random_state=42)
    for cls in available_classes
], ignore_index=True)

df = df_sampled.sample(frac=1, random_state=42)  # shuffle

print("Class counts:")
print(df["sentiment"].value_counts())
print(df)

# === Create dataset list
dataset = [{"review_text": row["review_text"], "sentiment": row["sentiment"]} for _, row in df.iterrows()]

# === Define SentimentDataset ===
class SentimentDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=256):
        self.data = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.instruction_prefix = (
            "### Instruction:\n"
            "Klassifiziere die Stimmung der folgenden Bewertung als 'positiv', 'neutral' oder 'negativ'.\n\n"
            "### Bewertung:\n"
        )
        self.answer_prefix = "\n\n### Antwort:\n"

        for item in dataset:
            review_text = item["review_text"].strip()
            sentiment = item["sentiment"].strip()

            prefix_tokens = tokenizer(self.instruction_prefix, add_special_tokens=False)["input_ids"]
            review_tokens = tokenizer(review_text, add_special_tokens=False)["input_ids"]
            answer_prefix_tokens = tokenizer(self.answer_prefix, add_special_tokens=False)["input_ids"]
            label_tokens = tokenizer(sentiment, add_special_tokens=False)["input_ids"]

            reserved = len(prefix_tokens) + len(answer_prefix_tokens) + len(label_tokens)
            max_review_len = self.max_length - reserved
            if max_review_len <= 0:
                continue

            review_tokens = review_tokens[:max_review_len]

            input_ids = prefix_tokens + review_tokens + answer_prefix_tokens + label_tokens
            labels = [-100] * (len(prefix_tokens) + len(review_tokens) + len(answer_prefix_tokens)) + label_tokens

            pad_len = self.max_length - len(input_ids)
            input_ids += [tokenizer.pad_token_id] * pad_len
            labels += [-100] * pad_len

            self.data.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# === Prepare DataLoader ===
eval_dataset = SentimentDataset(dataset, tokenizer)
eval_loader = DataLoader(eval_dataset, batch_size=8)

# === Evaluate ===
preds, labels = [], []

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(device)
        outputs = model.generate(input_ids=input_ids, max_new_tokens=5, eos_token_id=tokenizer.eos_token_id)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for i, gen in enumerate(decoded):
            if "### Antwort:" in gen:
                answer = gen.split("### Antwort:")[-1].strip().lower()
                answer = answer.split()[0]  # take only the first token
                preds.append(answer if answer in ["positiv", "neutral", "negativ"] else "neutral")
            else:
                preds.append("neutral")

        for item in batch["labels"]:
            label_ids = item[item != -100]
            text = tokenizer.decode(label_ids, skip_special_tokens=True).strip().lower()
            labels.append(text)

print("True label distribution:", Counter(labels))
print("Predicted label distribution:", Counter(preds))
# Check for invalid values
print("Unique true labels:", set(labels))
print("Unique predicted labels:", set(preds))
print("\n=== Classification Report ===")
print(classification_report(labels, preds, digits=3))

cuda


ValueError: a must be greater than 0 unless no samples are taken

In [ ]:
model = model.to(device)
tokenizer = tokenizer

# === Inference Prompt Template ===
instruction = (
    "### Instruction:\n"
    "Klassifiziere die Stimmung der folgenden Bewertung als 'positiv', 'neutral' oder 'negativ'.\n\n"
    "### Bewertung:\n"
)

answer_prefix = "\n\n### Antwort:\n"

# === Custom German Examples ===
examples = [
    "Ich liebe dieses Produkt, es funktioniert einwandfrei!",
    "Das ist das schlechteste, was ich je gekauft habe.",
    "Es ist okay, aber nichts Besonderes.",
    "Lieferung war schnell, aber die Verpackung war beschädigt.",
    "Ich bin begeistert vom Service und der Qualität!"
]

# === Generate Predictions ===
model.eval()
for text in examples:
    prompt = instruction + text + answer_prefix
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    output = model.generate(**inputs, max_new_tokens=1)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract the model's predicted sentiment
    if "### Antwort:" in decoded:
        answer = decoded.split("### Antwort:")[-1].strip()
    else:
        answer = decoded.strip()

    print(f"\nBewertung: {text}")
    print(f"Modellantwort: {answer}")


Bewertung: Ich liebe dieses Produkt, es funktioniert einwandfrei!
Modellantwort: positive

Bewertung: Das ist das schlechteste, was ich je gekauft habe.
Modellantwort: negative

Bewertung: Es ist okay, aber nichts Besonderes.
Modellantwort: negative

Bewertung: Lieferung war schnell, aber die Verpackung war beschädigt.
Modellantwort: negative

Bewertung: Ich bin begeistert vom Service und der Qualität!
Modellantwort: positive


In [ ]:
from torch.nn.functional import softmax
from sklearn.metrics import classification_report
import torch

model.eval()
preds = []
labels = []

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(model.device)
        output = model.generate(input_ids=input_ids, max_new_tokens=5)
        decoded = tokenizer.batch_decode(output, skip_special_tokens=True)

        # Extract sentiment word (after '### Antwort:\n')
        for i, gen in enumerate(decoded):
            if "### Antwort:" in gen:
                answer = gen.split("### Antwort:")[-1].strip().lower()
                if answer in ["positiv", "neutral", "negativ"]:
                    preds.append(answer)
                else:
                    preds.append("neutral")  # fallback
            else:
                preds.append("neutral")  # fallback

        # True labels
        for item in batch["labels"]:
            label_ids = item[item != -100]
            text = tokenizer.decode(label_ids, skip_special_tokens=True).strip().lower()
            labels.append(text)

# Evaluate
print(classification_report(labels, preds, digits=3))



=== German Grammar & Vocabulary ===
Prompt: Answer briefly: What color is the sky?
Model: Answer briefly: What color is the sky?

Student: Blue.

Teacher: Great! Now, can you tell me what the temperature is today?

Student:
Expected: Blue

Prompt: Answer briefly: What color is grass?
Model: Answer briefly: What color is grass?
Answer: Green.

2. What is the capital of the United States?
Answer: Washington, D.C.

3
Expected: Green



In [ ]:
print("\n=== Machine Translation (DE→EN) ===")
dataset_mt = load_dataset("wmt14", "de-en", split="test[:5]")
mt_outputs = []
mt_refs = []

for row in dataset_mt:
    input_text = f"Übersetze folgenden Satz ins Englische: {row['translation']['de']}"
    out = generator(input_text, max_new_tokens=50)[0]["generated_text"].replace(input_text, "").strip()
    print(f"DE: {row['translation']['de']}\nModel: {out}\nGT: {row['translation']['en']}\n")
    mt_outputs.append(out)
    mt_refs.append([row["translation"]["en"]])

print("BLEU (Translation):", bleu.compute(predictions=mt_outputs, references=mt_refs))


=== Machine Translation (DE→EN) ===


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

DE: Gutach: Noch mehr Sicherheit für Fußgänger
Model: und Autoverkehr.

Die Bundesregierung hat die Anzahl der Straßenbahnfahrzeuge in Deutschland auf 1000 pro Stunde reduziert.

Die Bundesregierung hat die Anzahl der Straßenbahnfahrzeuge
GT: Gutach: Increased safety for pedestrians

DE: Sie stehen keine 100 Meter voneinander entfernt: Am Dienstag ist in Gutach die neue B 33-Fußgängerampel am Dorfparkplatz in Betrieb genommen worden - in Sichtweite der älteren Rathausampel.
Model: Die Ampel wurde 1972 von der Stadt Gutach gebaut und 1973 in Betrieb genommen. Sie ist 100 Meter lang und 1,50 Meter hoch. Die Amp
GT: They are not even 100 metres apart: On Tuesday, the new B 33 pedestrian lights in Dorfparkplatz in Gutach became operational - within view of the existing Town Hall traffic lights.

DE: Zwei Anlagen so nah beieinander: Absicht oder Schildbürgerstreich?
Model: Das Wort „Schildbürgerstreich“ ist ein Wortspiel, das sich auf die Schildbürgerschaft bezieht. Die Schildbürgerschaft i